# 5th attempt - RNN

In [1]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [2]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
SIZE = 10
AMOUNT_BOARDS = 1000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 4

In [4]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE, gen)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 39520
len train:  32011
len val:  3557
len test:  3952


In [5]:
from functions import build_and_train_rnn

# build and train RNN model
model, history = build_and_train_rnn(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    rnn_units=128,
    dense_units=64,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

model.summary()


Epoch 1/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 25s 20ms/step - accuracy: 0.7920 - loss: 0.4645 - val_accuracy: 0.8112 - val_loss: 0.4061
Epoch 2/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.8148 - loss: 0.3939 - val_accuracy: 0.8124 - val_loss: 0.3954
Epoch 3/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - accuracy: 0.8297 - loss: 0.3714 - val_accuracy: 0.8204 - val_loss: 0.3848
Epoch 4/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - accuracy: 0.8477 - loss: 0.3349 - val_accuracy: 0.8266 - val_loss: 0.3834
Epoch 5/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.8680 - loss: 0.3002 - val_accuracy: 0.8199 - val_loss: 0.4081
Epoch 6/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.8942 - loss: 0.2527 - val_accuracy: 0.8146 - val_loss: 0.4347
Epoch 7/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 22s 16ms/step - accuracy: 0.9208 - loss: 0.1985 - val_accuracy: 0.8073 - val_loss: 0.5063
Epoch 8/20
801/801 ━━━━━━━━━━━━━━━━━━━━ 21s 16ms/step - accuracy: 0.9459 - loss: 0.1386 - 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 376,709 (1.44 MB)

 Trainable params: 125,569 (490.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 251,140 (981.02 KB)

In [6]:
X_test_rnn = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')

test_loss, test_acc = model.evaluate(X_test_rnn, y_test_array.astype('float32'))
print(f"Test accuracy: {test_acc:.3f}")

124/124 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7839 - loss: 1.7710
Test accuracy: 0.795


In [8]:
evaluate_model(model, X_test_rnn, y_test_array, SIZE, gen)

124/124 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        401 │        421 │
│ True=Dead    │        391 │       2739 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.795
Precision   : 0.506
Recall      : 0.488
F1-score    : 0.497
